In [46]:
import pandas as pd
import numpy as np

train_df=pd.read_csv('/content/train_LZdllcl.csv')
test_df=pd.read_csv('/content/test_2umaH9m.csv')

## Exploratory Data Analysis (EDA)

In [47]:
print("Train data info:")
train_df.info()

Train data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54808 entries, 0 to 54807
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   employee_id           54808 non-null  int64  
 1   department            54808 non-null  object 
 2   region                54808 non-null  object 
 3   education             52399 non-null  object 
 4   gender                54808 non-null  object 
 5   recruitment_channel   54808 non-null  object 
 6   no_of_trainings       54808 non-null  int64  
 7   age                   54808 non-null  int64  
 8   previous_year_rating  50684 non-null  float64
 9   length_of_service     54808 non-null  int64  
 10  KPIs_met >80%         54808 non-null  int64  
 11  awards_won?           54808 non-null  int64  
 12  avg_training_score    54808 non-null  int64  
 13  is_promoted           54808 non-null  int64  
dtypes: float64(1), int64(8), object(5)
memory usage: 5.9+

In [48]:
print("Missing values in train data:")
train_df.isnull().sum()

Missing values in train data:


,0
employee_id,0
department,0
region,0
education,2409
gender,0
recruitment_channel,0
no_of_trainings,0
age,0
previous_year_rating,4124
length_of_service,0


In [49]:
# Separating features and target

X_train_full=train_df.drop(['employee_id', 'is_promoted'], axis=1)
y_train_full=train_df['is_promoted']

In [50]:
X_test=test_df.drop(['employee_id'], axis=1)

## Pre-processing

In [51]:
# Filling missing categorical with mode, numerical with median for train data

train_edu_mode=X_train_full['education'].mode()[0]
train_rating_median=X_train_full['previous_year_rating'].median()

In [14]:
X_train_full['education']=X_train_full['education'].fillna(train_edu_mode)
X_train_full['previous_year_rating']=X_train_full['previous_year_rating'].fillna(train_rating_median)

In [52]:
# Fill missing categorical with mode, numerical with median for test data

test_edu_mode=X_test['education'].mode()[0]
test_rating_median=X_test['previous_year_rating'].median()

In [53]:
X_test['education']=X_test['education'].fillna(test_edu_mode)
X_test['previous_year_rating']=X_test['previous_year_rating'].fillna(test_rating_median)

In [54]:
# Encoding categorical values

cat_cols=['department', 'region', 'education', 'gender', 'recruitment_channel']
label_enc={}

In [55]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
for col in cat_cols:
    le=LabelEncoder()
    X_train_full[col]=le.fit_transform(X_train_full[col])
    X_test[col]=le.transform(X_test[col])

In [56]:
# Scaling the values

scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train_full)
X_test_scaled=scaler.transform(X_test)

## Modelling

In [57]:
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(X_train_scaled, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full)

## Fine-tuning and modelling

In [58]:
# Calculating the imbalance weight

sp_weight=(y_tr==0).sum()/(y_tr==1).sum()

In [59]:
sp_weight

np.float64(10.742367434386717)

In [60]:
# Using XGBoost as the base model

from xgboost import XGBClassifier
xgb_model=XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    scale_pos_weight=sp_weight,
    random_state=42,
    eval_metric='logloss'
)

In [61]:
xgb_model.fit(X_tr, y_tr)
xgb_val_probs=xgb_model.predict_proba(X_val)[:, 1]

In [62]:
# Training the LightGBM model

from lightgbm import LGBMClassifier
lgbm_model=LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

In [63]:
lgbm_model.fit(X_tr, y_tr)
lgbm_val_probs=lgbm_model.predict_proba(X_val)[:, 1]

[LightGBM] [Info] Number of positive: 3734, number of negative: 40112
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004353 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 219
[LightGBM] [Info] Number of data points in the train set: 43846, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [64]:
# Mixing the models

mixed_val=(xgb_val_probs+lgbm_val_probs)/2

In [67]:
# Tune thresholding

best_threshold=0.5
best_f1=0

for thresh in np.arange(0.1, 0.9, 0.01):
    preds=(mixed_val>=thresh).astype(int)
    score=f1_score(y_val, preds)
    if score>best_f1:
        best_f1=score
        best_threshold=thresh

print(f" Validation F1: {best_f1:.4f} (at threshold: {best_threshold:.2f})")

 Validation F1: 0.5176 (at threshold: 0.85)


In [68]:
# Complete data training

full_sp_weight=(y_train_full==0).sum()/(y_train_full==1).sum()
xgb_model.set_params(scale_pos_weight=full_sp_weight)

xgb_model.fit(X_train_scaled, y_train_full)
lgbm_model.fit(X_train_scaled, y_train_full)

xgb_test_probs=xgb_model.predict_proba(X_test_scaled)[:, 1]
lgbm_test_probs=lgbm_model.predict_proba(X_test_scaled)[:, 1]

final_blended_probs=(xgb_test_probs+lgbm_test_probs)/2
final_preds=(final_blended_probs>=best_threshold).astype(int)

[LightGBM] [Info] Number of positive: 4668, number of negative: 50140
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002131 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 220
[LightGBM] [Info] Number of data points in the train set: 54808, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [69]:
# Replacing the target column with the prediction values you get using the test dataset for your best model

sample=pd.read_csv('/content/sample_submission_M0L0uXE.csv')
sample['is_promoted']=final_preds
sample.to_csv('final_submission.csv', index=False)